# Market Position Based Pricing

Manual-run review script. For each (product, cohort), classify the SKU by its position in the market percentile band and yesterday's achievement bucket (`qty_ratio = yest_qty / p80_target`), then move N steps in the SKU's effective tier ladder per the agreed action matrix.

**No API push.** Outputs a timestamped Excel for review.

## Action matrix

| pos | low (<0.9) | mid (0.9-2.0) | high (>=2.0) |
|---|---|---|---|
| min | HOLD | HOLD | +2 |
| 25 | -1 | -1 | +1 |
| 50 | -2 | -1 | +1 |
| 75 | -3 | -3 | HOLD |
| max | -3 | -2 | HOLD |
| Target margin<target | -2 | -2 | +2 |
| Target margin>=target | -2 | -1 | HOLD |
| below_min | exact `commercial_min` | exact `commercial_min` | exact `commercial_min` |

Margin step bound: `0.1 * target_margin <= |delta_margin| <= 0.5 * target_margin`. Hard floor: `0.9 * wac_p` (commercial_min takes precedence).

In [12]:
# =============================================================================
# CELL 1 - IMPORTS + SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake, get_snowflake_timezone
from constants import WAREHOUSE_MAPPING, COHORT_IDS

TIMEZONE = get_snowflake_timezone()
CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}  |  Snowflake TZ: {TIMEZONE}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Ready. Today: 2026-04-29  |  Snowflake TZ: America/Los_Angeles


In [13]:
# =============================================================================
# CELL 2 - SOURCE DATA LOAD
# Copies the loader pattern from manual_price_push.ipynb but with two important
# differences:
#   (a) current_price comes from cohort_pricing_changes only (DBDP is stopped).
#   (b) margin tiers are loaded via get_margin_tiers() so the Target case can
#       fall back to wac/(1 - margin_tier_X) prices.
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

# (a) current_price from cohort_pricing_changes (latest row per cohort, product, PU)
print('Loading current prices (cohort_pricing_changes)...')
CURRENT_PRICES_QUERY = f'''
WITH latest_changes AS (
    SELECT
        cpc.cohort_id,
        pup.product_id,
        pup.packing_unit_id,
        pup.basic_unit_count,
        cpc.price AS current_price
    FROM cohort_pricing_changes cpc
    JOIN packing_unit_products pup ON pup.id = cpc.product_packing_unit_id
    WHERE cpc.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
      AND pup.basic_unit_count = 1
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cpc.cohort_id, pup.product_id, pup.packing_unit_id
        ORDER BY cpc.created_at DESC
    ) = 1
)
SELECT cohort_id, product_id, packing_unit_id, basic_unit_count, current_price
FROM latest_changes
WHERE basic_unit_count = 1
  AND ((product_id = 1309 AND packing_unit_id = 2) OR product_id <> 1309)
'''
df_prices = query_snowflake(CURRENT_PRICES_QUERY)
print(f'  Current prices: {len(df_prices)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins (brand x cat)...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading commercial min prices...')
COMMERCIAL_MIN_QUERY = f'''
WITH to_remove AS (
    SELECT check_date AS start_date,
           (check_date + INTERVAL '1 month') + 6 AS end_date
    FROM (
        SELECT CASE
                 WHEN DATE_PART('day', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) < 7
                 THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
                 ELSE DATE_FROM_PARTS(
                       YEAR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE),
                       MONTH(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE),
                       1)
               END AS check_date
    )
),
region_mapping AS (
    SELECT * FROM (VALUES
        ('Greater Cairo', 'Cairo'),
        ('Greater Cairo', 'Giza')
    ) x(region, main_region)
),
warehouse_mapping AS (
    SELECT * FROM (VALUES
        ('Cairo', 'Mostorod', 1, 700),
        ('Giza', 'Barageel', 236, 701),
        ('Giza', 'Sakkarah', 962, 701),
        ('Delta West', 'El-Mahala', 337, 703),
        ('Delta West', 'Tanta', 8, 703),
        ('Delta East', 'Mansoura FC', 339, 704),
        ('Delta East', 'Sharqya', 170, 704),
        ('Upper Egypt', 'Assiut FC', 501, 1124),
        ('Upper Egypt', 'Bani sweif', 401, 1126),
        ('Upper Egypt', 'Menya Samalot', 703, 1123),
        ('Upper Egypt', 'Sohag', 632, 1125),
        ('Alexandria', 'Khorshed Alex', 797, 702)
    ) x(region, warehouse_name, warehouse_id, cohort_id)
)
SELECT DISTINCT
    sku_id AS product_id,
    cohort_id,
    min_price AS commercial_min_price
FROM (
    SELECT mp.product_id AS sku_id, mp.region, min_price, wac1, created_at,
           MAX(created_at) OVER (PARTITION BY mp.product_id, mp.region) AS latest_date
    FROM finance.minimum_prices mp
    JOIN finance.all_cogs f ON f.product_id = mp.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    WHERE is_deleted = 'false'
      AND created_at BETWEEN (SELECT start_date FROM to_remove) AND (SELECT end_date FROM to_remove)
) comm
LEFT JOIN region_mapping rm ON comm.region = rm.region
LEFT JOIN warehouse_mapping wm ON wm.region = COALESCE(rm.main_region, comm.region)
WHERE created_at = latest_date
  AND min_price < wac1 * 1.5
'''
df_commercial_min = query_snowflake(COMMERCIAL_MIN_QUERY)
print(f'  Commercial min prices: {len(df_commercial_min)} rows')

print('Loading stocks (per warehouse, basic unit)...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
      AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

# (b) Margin tiers per (warehouse_id, product_id) - margin VALUES, not prices.
print('Loading margin tiers...')
df_margin_tiers = get_margin_tiers()
print(f'  Margin tiers: {len(df_margin_tiers)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()

In [14]:
# =============================================================================
# CELL 3 - BUILD LOOKUP (product x cohort) + margin-tier merge
# =============================================================================

# Step 3a - same shape as manual_price_push lookup
df_market_cohorts = expand_to_cohorts(market_data)

lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# brand x cat target margin fallback (and 0.05 default)
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_margin_tgt', 'target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns
                     if c.startswith('target_bm') or c.startswith('cat_target') or c == 'target_margin_tgt'],
            inplace=True, errors='ignore')

# market percentile derivatives from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# current price per (product, cohort) - basic unit
base_prices = (
    df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']]
    .drop_duplicates(subset=['cohort_id', 'product_id'])
)
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# commercial_min_price per (product, cohort)
if 'cohort_id' in df_commercial_min.columns:
    cmin = df_commercial_min[['product_id', 'cohort_id', 'commercial_min_price']].drop_duplicates()
    lookup = lookup.merge(cmin, on=['product_id', 'cohort_id'], how='left')
else:
    lookup['commercial_min_price'] = np.nan

# total stocks per product (display only)
product_stocks = (
    df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    .rename(columns={'stocks': 'total_stocks'})
)
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With current_price: {lookup["current_price"].notna().sum()}')
print(f'  With commercial_min: {lookup["commercial_min_price"].notna().sum()}')

# Step 3b - margin tiers, warehouse -> cohort by MEAN of margin values
margin_tier_cols = [
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
]

wh_to_cohort = pd.DataFrame(
    WAREHOUSE_MAPPING,
    columns=['region', 'wh_name', 'warehouse_id', 'cohort_id'],
)[['warehouse_id', 'cohort_id']]

_present_tier_cols = [c for c in margin_tier_cols if c in df_margin_tiers.columns]
df_margin_tiers_cohort = (
    df_margin_tiers.merge(wh_to_cohort, on='warehouse_id', how='inner')
                   .groupby(['cohort_id', 'product_id'], as_index=False)[_present_tier_cols]
                   .mean()
)
lookup = lookup.merge(
    df_margin_tiers_cohort,
    on=['cohort_id', 'product_id'],
    how='left',
)
if 'margin_tier_1' in lookup.columns:
    print(f'  With margin_tier_1: {lookup["margin_tier_1"].notna().sum()}')

Lookup table: 234702 rows (product x cohort)
  With market data: 44134
  With WAC: 76149
  With current_price: 74542
  With commercial_min: 636
  With margin_tier_1: 26385


In [15]:
# =============================================================================
# CELL 4 - ACHIEVEMENT JOIN (yesterday)
# Roll up warehouse to cohort by summing yest_qty + p80_target across the
# cohort's warehouses, then recompute qty_ratio at cohort level.
# =============================================================================
ACHIEVMENT_QUERY = open(os.path.join('queries', 'achievment_yasterday.sql'), encoding='utf-8').read()
df_ach = query_snowflake(ACHIEVMENT_QUERY)
print(f'Achievement rows (warehouse, product): {len(df_ach)}')

# achievment_yasterday returns warehouse_id, product_id, ... .
# Roll up to cohort using WAREHOUSE_MAPPING (already loaded above as wh_to_cohort).
#df_ach_w = df_ach.merge(wh_to_cohort, on='warehouse_id', how='inner')

df_ach_cohort = (
    df_ach.groupby(['cohort_id', 'product_id'], as_index=False)
            .agg(yest_qty=('yest_qty', 'sum'),
                 p80_target=('p80_target', 'sum'),
                 opening_stock=('opening_stock', 'sum'),
                 closing_stock=('closing_stock', 'sum'))
)
df_ach_cohort['qty_ratio'] = (
    df_ach_cohort['yest_qty'] /
    df_ach_cohort['p80_target'].replace(0, np.nan)
)

lookup = lookup.merge(df_ach_cohort, on=['cohort_id', 'product_id'], how='left')
lookup['qty_ratio'] = lookup['qty_ratio'].fillna(0)
print(f'Lookup with achievement: {len(lookup)} rows; '
      f'{lookup["closing_stock"].notna().sum()} have closing_stock')

Achievement rows (warehouse, product): 16873
Lookup with achievement: 234702 rows; 12753 have closing_stock


In [16]:
# =============================================================================
# CELL 5 - FILTER
# Keep only SKUs that had inventory at both opening and closing of yesterday.
# Achievement query already filters out warehouses with closing or opening = 0,
# so SKUs with NaN closing/opening here were dropped upstream.
# =============================================================================
before = len(lookup)
lookup = lookup[
    (lookup['closing_stock'].fillna(0) > 0) &
    (lookup['opening_stock'].fillna(0) > 0)
].copy()
print(f'Filtered to closing>0 AND opening>0: {before} -> {len(lookup)}')

Filtered to closing>0 AND opening>0: 234702 -> 12753


In [17]:
# =============================================================================
# CELL 6 - POSITION + ACH BUCKET
# =============================================================================
def derive_position(row):
    cur = row.get('current_price')
    cmin = row.get('commercial_min_price', 0) or 0
    if pd.isna(cur):
        return 'no_price'
    if cmin > 0 and cur < cmin:
        return 'below_min'
    has_market = (
        isinstance(row.get('price_tiers'), list) and len(row['price_tiers']) > 0
        and pd.notna(row.get('market_min'))
    )
    if not has_market:
        return 'Target'
    if cur >= row['market_max']: return 'max'
    if cur >= row['market_75']:  return '75'
    if cur >= row['market_50']:  return '50'
    if cur >= row['market_25']:  return '25'
    return 'min'

def derive_ach_bucket(ach):
    if pd.isna(ach) or ach < 0.9: return 'low'
    if ach < 2.0:                  return 'mid'
    return 'high'

lookup['position']   = lookup.apply(derive_position, axis=1)
lookup['ach_bucket'] = lookup['qty_ratio'].apply(derive_ach_bucket)

print('Position x Ach distribution:')
print(pd.crosstab(lookup['position'], lookup['ach_bucket'], margins=True))

Position x Ach distribution:
ach_bucket  high    low   mid    All
position                            
25           148    778   423   1349
50            83    622   293    998
75            37    429   129    595
Target        27   1750    92   1869
below_min      0     53     0     53
max           24     85    99    208
min          172   6856   653   7681
All          491  10573  1689  12753


In [18]:
# =============================================================================
# CELL 7 - ACTION MATRIX
#   - build_effective_tiers: price_tiers if non-empty, else margin tier prices.
#   - step_in_tiers: move N steps relative to current_price (clamps to ends).
#   - apply_margin_step_bounds: 0.1*tgt <= |delta_margin| <= 0.5*tgt.
#   - compute_action: per-row matrix dispatch.
# Safety nets at the end: 0.9*wac_p hard floor, then commercial_min_price.
# =============================================================================
def build_effective_tiers(row):
    """Sorted ascending list of prices (0.25 EGP rounded).
    Per design: margin MEANs are pre-aggregated at cohort grain in Cell 3, then
    converted to ONE price per tier via wac/(1-mean_margin)."""
    pt = row.get('price_tiers')
    if isinstance(pt, list) and len(pt) > 0:
        return sorted({round(p * 4) / 4 for p in pt if p > 0})
    wac = row.get('wac_p', 0) or 0
    if wac <= 0:
        return []
    margins = []
    for c in ['margin_tier_1','margin_tier_2','margin_tier_3','margin_tier_4',
              'margin_tier_5','margin_tier_above_1','margin_tier_above_2']:
        m = row.get(c)
        if pd.notna(m) and 0 < m < 1:
            margins.append(m)
    return sorted({round(wac / (1 - m) * 4) / 4 for m in margins})


def step_in_tiers(tiers, current, n):
    """Returns (new_price, was_clamped, no_tier_in_direction)."""
    if n == 0 or not tiers:
        return None, False, False
    if n > 0:
        above = [t for t in tiers if t > current]
        if not above:
            return None, False, True
        idx = min(n - 1, len(above) - 1)
        return above[idx], (idx < n - 1), False
    below = [t for t in tiers if t < current][::-1]
    if not below:
        return None, False, True
    idx = min(abs(n) - 1, len(below) - 1)
    return below[idx], (idx < abs(n) - 1), False


def apply_margin_step_bounds(current, candidate, wac, target_margin):
    """Clamp |delta_margin| to [0.1*tgt, 0.5*tgt]. Returns (price, bound_hit)."""
    if candidate is None or wac <= 0 or current <= 0:
        return candidate, None
    cur_m = (current   - wac) / current
    new_m = (candidate - wac) / candidate
    delta = new_m - cur_m
    if delta == 0:
        return candidate, None
    direction = 1 if delta > 0 else -1
    abs_delta = abs(delta)
    min_step  = 0.1 * (target_margin or 0)
    max_step  = 0.5 * (target_margin or 0)
    if min_step <= 0 and max_step <= 0:
        return candidate, None
    if abs_delta < min_step:
        clamped_m = cur_m + direction * min_step
        bound = 'min'
    elif abs_delta > max_step:
        clamped_m = cur_m + direction * max_step
        bound = 'max'
    else:
        return candidate, None
    if clamped_m >= 0.99:
        return candidate, None
    new_price = wac / (1 - clamped_m)
    return round(new_price * 4) / 4, bound


ACTION_STEPS = {
    ('min', 'low'):  0,    # 1
    ('min', 'mid'):  0,    # 2
    ('min', 'high'): +2,   # 3
    ('25',  'low'):  -1,   # 4
    ('25',  'mid'):  -1,   # 5
    ('25',  'high'): +1,   # 6
    ('50',  'low'):  -2,   # 7
    ('50',  'mid'):  -1,   # 8
    ('50',  'high'): +1,   # 9
    ('75',  'low'):  -3,   # 10
    ('75',  'mid'):  -3,   # 11
    ('75',  'high'): 0,    # 12
    ('max', 'low'):  -3,   # 13
    ('max', 'mid'):  -2,   # 14
    ('max', 'high'): 0,    # 15
}


def compute_action(row):
    pos  = row['position']
    ach  = row['ach_bucket']
    cur  = row['current_price']
    cmin = row.get('commercial_min_price', 0) or 0
    wac  = row.get('wac_p', 0) or 0
    tm   = row.get('target_margin', 0.05) or 0.05

    if pos == 'no_price':
        return None, 'no_action_no_price', None
    if pos == 'below_min':
        return round(cmin * 4) / 4, 'lift_to_commercial_min', None

    tiers = build_effective_tiers(row)

    if pos == 'Target':
        cur_margin = (cur - wac) / cur if cur > 0 and wac > 0 else 0
        margin_below = cur_margin < tm
        if margin_below:
            n = +2 if ach == 'high' else -2
            base_label = f'target_marg_below_{ach}_{n:+d}'
        else:
            n = {'low': -2, 'mid': -1, 'high': 0}[ach]
            base_label = (f'target_marg_ok_{ach}_{n:+d}'
                          if n != 0 else f'target_marg_ok_{ach}_hold')
    else:
        n = ACTION_STEPS.get((pos, ach), None)
        base_label = (f'pos_{pos}_{ach}_{n:+d}'
                      if n not in (None, 0) else f'pos_{pos}_{ach}_hold')

    if n in (None, 0):
        return None, base_label, n

    candidate, clamped, no_tier = step_in_tiers(tiers, cur, n)
    if no_tier:
        return None, f'{base_label}_no_tier_{"above" if n > 0 else "below"}', n

    label = base_label + ('_clamped' if clamped else '')

    bounded, bound_hit = apply_margin_step_bounds(cur, candidate, wac, tm)
    if bound_hit:
        label += f'_step_{bound_hit}_bound'
        candidate = bounded

    return candidate, label, n


results = lookup.apply(compute_action, axis=1, result_type='expand')
results.columns = ['new_price', 'reason', 'steps']
lookup = pd.concat(
    [lookup.drop(columns=['new_price', 'reason', 'steps'], errors='ignore'), results],
    axis=1,
)

# Safety net 1: never go below 0.9 * wac_p
mask = lookup['new_price'].notna() & (lookup['wac_p'] > 0)
wac_floor = 0.9 * lookup['wac_p']
below_wac = mask & (lookup['new_price'] < wac_floor)
lookup.loc[below_wac, 'new_price'] = (wac_floor[below_wac] * 4).round() / 4

# Safety net 2: commercial_min takes precedence when present
mask  = lookup['new_price'].notna() & (lookup['commercial_min_price'].fillna(0) > 0)
below = mask & (lookup['new_price'] < lookup['commercial_min_price'])
lookup.loc[below, 'new_price'] = lookup.loc[below, 'commercial_min_price'].round(2)

print('Reason distribution (top 25):')
print(lookup['reason'].value_counts().head(25))

Reason distribution (top 25):
reason
pos_min_low_hold                                   6856
target_marg_below_low_-2_no_tier_below             1174
pos_min_mid_hold                                    653
pos_50_low_-2                                       535
pos_25_low_-1                                       465
pos_25_low_-1_step_min_bound                        298
pos_25_mid_-1                                       292
target_marg_below_low_-2                            258
pos_75_low_-3                                       229
pos_50_mid_-1                                       204
pos_75_low_-3_step_max_bound                        171
pos_min_high_+2                                     137
pos_25_mid_-1_step_min_bound                        127
pos_25_high_+1                                       93
pos_50_mid_-1_step_min_bound                         88
pos_max_mid_-2                                       86
pos_75_mid_-3                                        73
target_marg

In [19]:
# =============================================================================
# CELL 8 - REVIEW EXPORT
# =============================================================================
to_review = lookup[
    lookup['new_price'].notna() &
    (lookup['new_price'] != lookup['current_price'])
].copy()
to_review['delta']     = to_review['new_price'] - to_review['current_price']
to_review['delta_pct'] = to_review['delta'] / to_review['current_price']

cols = [
    'cohort_id', 'product_id', 'sku', 'brand', 'cat',
    'wac_p', 'target_margin', 'commercial_min_price',
    'price_tiers',
    'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'market_avg',
    'current_price', 'qty_ratio', 'position', 'ach_bucket', 'steps',
    'opening_stock', 'closing_stock', 'total_stocks',
    'reason', 'new_price', 'delta', 'delta_pct',
]
to_review = to_review[[c for c in cols if c in to_review.columns]]

print(f'Total SKUs with action: {len(to_review)}')
print('Reason distribution:')
print(to_review['reason'].value_counts())
print('\nPosition x Ach distribution:')
print(pd.crosstab(to_review['position'], to_review['ach_bucket'], margins=True))

stamp = datetime.now(CAIRO_TZ).strftime('%Y%m%d_%H%M')
out_path = f'market_position_pricing_review_{stamp}.xlsx'
to_review.to_excel(out_path, index=False, engine='xlsxwriter')
print(f'\nReview file written: {out_path}')

Total SKUs with action: 3556
Reason distribution:
reason
pos_50_low_-2                                      496
pos_25_low_-1                                      436
pos_25_mid_-1                                      292
pos_25_low_-1_step_min_bound                       270
pos_75_low_-3                                      229
pos_50_mid_-1                                      204
pos_75_low_-3_step_max_bound                       171
pos_min_high_+2                                    137
pos_25_mid_-1_step_min_bound                       127
pos_25_high_+1                                      93
pos_50_mid_-1_step_min_bound                        88
pos_max_mid_-2                                      86
pos_75_mid_-3                                       73
target_marg_below_low_-2_clamped_step_min_bound     65
pos_50_high_+1                                      64
pos_25_high_+1_step_min_bound                       55
pos_75_mid_-3_step_max_bound                        54
lift_to_